# Codify 301a: configuration changes the output

Nothing about a jurisdiction is hard-coded: the markers the scan looks for, the
hierarchy the AKN carries, the citation form, the calendar, are all read from
`data/jurisdictions/<code>/config.json`. This notebook puts one short act through
several configurations, then edits a rule and shows what moved.

Sections 1 to 3 and the scans in section 5 run without a model. Section 4 makes three
short structuring calls and section 5 ends with one more, a cent or two in all; both need
the `.env` from [Codify 101](codify-101.ipynb).

In [1]:
import logging
from pathlib import Path

import langfuse  # noqa: F401, its import resets the logger quieted below
import structlog
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # the repo's .env; exported variables win
logging.getLogger("langfuse").setLevel(logging.ERROR)  # tracing is optional
structlog.configure(  # warnings only, uncoloured
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
    processors=[structlog.processors.add_log_level, structlog.dev.ConsoleRenderer(colors=False)],
)
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
FIXTURES = REPO / "tests" / "fixtures" / "synthetic"

## 1. What a configuration declares

Five bundled configurations: two synthetic (`xa` common-law English, `xl` Indonesian
civil-law) and three public-reference ones.

In [2]:
import pandas as pd

from codify.jurisdictions import load_config

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)

CODES = ["xa", "gb", "ie", "ee", "xl"]
configs = {code: load_config(code) for code in CODES}


def describe(cfg):
    dc = cfg.document_classes[cfg.default_document_class]
    return {
        "name": cfg.name_en or cfg.name,
        "languages": ",".join(cfg.languages),
        "calendar": cfg.calendar,
        "basic unit": dc.basic_unit,
        "hierarchy": " > ".join(h.local_term for h in dc.hierarchy if h.level != "grouping"),
        "citation": cfg.numbering.act_citation if cfg.numbering else None,
        "eras": len(cfg.legal_eras),
    }


pd.DataFrame({code: describe(cfg) for code, cfg in configs.items()}).T

,name,languages,calendar,basic unit,hierarchy,citation,eras
xa,Commonwealth of Atlantis,eng,gregorian,section,Part > Chapter > Section > Subsection > Paragraph > Subparagraph,{Short Title},2
gb,United Kingdom of Great Britain and Northern Ireland,eng,gregorian,section,Part > Chapter > Section > Subsection > Paragraph > Subparagraph,{Short Title} {year},0
ie,Ireland,"eng,gle",gregorian,section,Part > Chapter > Section > Subsection > Paragraph > Subparagraph,Number {number} of {year},0
ee,Estonia,est,gregorian,article,Osa > Peatükk > Jagu > Alljaotus / Alajagu > § (Paragrahv) > Lõige > Punkt > Alapunkt,"RT I, {date}, {number}",0
xl,Republic of Langkasuka,ind,gregorian,article,Buku (Book) > Bab (Chapter) > Bagian (Part/Section) > Paragraf > Pasal (Article) > Ayat (Verse/Clause) > Huruf (Letter) > Angka (Number),UU {number}/{year},0


## 2. One text, five scans

The same English act scanned under every configuration, and an Indonesian rendering of
it. The scan is deterministic and needs no model: it is the half of structuring that
decides what the model will be asked to fill.

In [3]:
from codify.quality.corpus_scan import scan_text

EN = """\
No. 9 of 2015

Coastal Lights Act, 2015

An Act to provide for the maintenance of coastal lights and for connected purposes.

PART I
PRELIMINARY

Section 1
This Act may be cited as the Coastal Lights Act.

Section 2
In this Act "light" means a lighthouse, beacon or buoy maintained under this Act.

PART II
THE KEEPER OF LIGHTS

Section 3
The Minister shall appoint a Keeper of Lights for a term of five years.

Section 4
A person who obscures or removes a light without the Keeper's consent commits an offence.
"""

ID = """\
UNDANG-UNDANG NOMOR 9 TAHUN 2015

TENTANG MERCUSUAR PANTAI

BAB I
KETENTUAN UMUM

Pasal 1
Undang-Undang ini disebut Undang-Undang Mercusuar Pantai.

Pasal 2
Dalam Undang-Undang ini "mercusuar" berarti menara suar, rambu atau pelampung yang
dipelihara berdasarkan Undang-Undang ini.

BAB II
PENJAGA MERCUSUAR

Pasal 3
Menteri mengangkat seorang Penjaga Mercusuar untuk masa jabatan lima tahun.

Pasal 4
Setiap orang yang menutupi atau memindahkan mercusuar tanpa persetujuan Penjaga dipidana.
"""


def scanned(code, text):
    cfg = configs[code]
    s = scan_text(text, config=cfg, country=code, doctype=cfg.default_document_class)
    return {"anchors": s.anchors, "by kind": s.by_kind, "basic units": s.basic_units}


rows = {f"{code} / English": scanned(code, EN) for code in CODES}
rows["xl / Indonesian"] = scanned("xl", ID)
rows["xa / Indonesian"] = scanned("xa", ID)
pd.DataFrame(rows).T

,anchors,by kind,basic units
xa / English,6,"{'part': 2, 'section': 4}",4
gb / English,6,"{'part': 2, 'section': 4}",4
ie / English,6,"{'part': 2, 'section': 4}",4
ee / English,6,"{'part': 2, 'section': 4}",0
xl / English,6,{'part': 6},0
xl / Indonesian,6,"{'article': 4, 'chapter': 2}",4
xa / Indonesian,0,{},0


Three configurations read the English act the same way. `ee` finds the same markers but
counts no basic unit, because its basic unit is the `§`; `xl` reads `Section 1` as a
part heading and finds no article at all, and `xa` finds nothing in the Indonesian text.
A scan with no basic units is the case the pipeline logs as `body_fill_skipped`: the
skeleton is exported and nothing is filled, which is what a wrong configuration looks
like from the outside.

## 3. Calendar and era

A configuration can declare legal eras, each with its own enacting authority and
calendar. `xa` has a pre-1847 era on the lunar Hijri calendar; a year read from a
document in that era converts before it reaches the FRBR URI.

In [4]:
from codify.calendar import year_from_calendar
from codify.quality.structural_scan import era_of

xa = configs["xa"]
for era in xa.legal_eras:
    print(f"  {era.id:<14} {era.enacting_authority:<36} calendar {era.calendar or xa.calendar}")
print("1840 ->", era_of(xa, 1840), "| 2015 ->", era_of(xa, 2015))
print("Hijri 1256 ->", year_from_calendar("1256", "lunar_hijri"))

  thalassocracy  Warden of the Concentric Harbours    calendar lunar_hijri
  assembly       Assembly of Atlantis                 calendar gregorian
1840 -> thalassocracy | 2015 -> assembly
Hijri 1256 -> 1840


## 4. Structure it under three configurations

The full pass, as in 101, for the English act under `xa` and `ie` and the Indonesian
act under `xl`. What differs is not the code path but what the configuration told it:
element names, eIds and the country segment of the FRBR URI.

In [5]:
import os

from codify.akn import parse_akn
from codify.core.llm import create_llm_client
from codify.pipeline.events import Complete, Failed
from codify.pipeline.formats.pdf import ingest_text

llm = create_llm_client(
    base_url=os.environ["LITELLM_BASE_URL"],
    api_key=os.environ["LITELLM_API_KEY"],
    model=os.environ.get("LITELLM_MODEL", "gemini-3.7-flash"),
    telemetry_mode="direct",
)


async def structure(code, text):
    akn_xml = ""
    async for event in ingest_text(text, code, llm=llm, name="coastal-lights"):
        if isinstance(event, Failed):
            raise RuntimeError(f"{event.stage}: {event.error}")
        if isinstance(event, Complete):
            akn_xml = event.akn_xml
    return parse_akn(akn_xml)


def eids(elements):
    for el in elements:
        yield el.akn_eid
        yield from eids(el.children)


docs = {}
for code, text in [("xa", EN), ("ie", EN), ("xl", ID)]:
    docs[code] = await structure(code, text)
pd.DataFrame(
    {
        code: {
            "work URI": d.frbr_work_uri,
            "expression URI": d.frbr_expression_uri,
            "language": d.language,
            "top level": ", ".join(sorted({el.akn_type for el in d.body})),
            "basic unit": ", ".join(sorted({el.akn_type for top in d.body for el in top.children})),
            "eIds": " ".join(eids(d.body)),
        }
        for code, d in docs.items()
    }
).T

[warning  ] body_fill_unowned_targets      chunk=w0 eids=['part_II']


[warning  ] body_fill_unowned_targets      chunk=w0 eids=['part_II']


[warning  ] body_fill_unowned_targets      chunk=w0 eids=['chp_II']


,work URI,expression URI,language,top level,basic unit,eIds
xa,/akn/xa/act/2015/9,/akn/xa/act/2015/9/eng@2015-01-01,eng,part,section,part_I part_I__sec_1 part_I__sec_2 part_II part_II__sec_3 part_II__sec_4
ie,/akn/ie/act/2015/9,/akn/ie/act/2015/9/eng@2015-01-01,eng,part,section,part_I part_I__sec_1 part_I__sec_2 part_II part_II__sec_3 part_II__sec_4
xl,/akn/xl/act/2015/9,/akn/xl/act/2015/9/ind@2015-01-01,ind,chapter,article,chp_I chp_I__art_1 chp_I__art_2 chp_II chp_II__art_3 chp_II__art_4


## 5. Change one rule

Configurations are data, so a change is a copy and an edit. `CODIFY_DATA_ROOT` points
the library at another `data/` tree; it is read at import, so the runs below go through
the command line in a subprocess. The edit: `xa`'s basic unit is spelt `Clause` instead
of `Section`, as some older statute books do. The text uses `Clause`; the stock
configuration does not know the word.

In [6]:
import json
import shutil
import subprocess
import sys
import tempfile

scratch = Path(tempfile.mkdtemp())
data_root = scratch / "data"
shutil.copytree(REPO / "data", data_root)
config_path = data_root / "jurisdictions" / "xa" / "config.json"
config = json.loads(config_path.read_text())
for level in config["document_classes"]["act"]["hierarchy"]:
    if level["level"] == "basic":
        level["local_term"] = "Clause"
config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False))

corpus = scratch / "corpus"
corpus.mkdir()
(corpus / "coastal-lights.txt").write_text(EN.replace("Section", "Clause"))

CODIFY = Path(sys.executable).parent / "codify"
EDITED = {"CODIFY_DATA_ROOT": str(data_root)}


def run(*args, env):
    done = subprocess.run(  # noqa: S603, a fixed argv
        [CODIFY, *args], env={**os.environ, **env}, cwd=REPO, capture_output=True, text=True
    )
    if done.returncode:
        raise RuntimeError(done.stderr[-2000:])


def scan_corpus(env):
    rows = scratch / "rows.jsonl"
    run("scan-corpus", corpus, "--jurisdiction", "xa", "--per-document", rows, env=env)
    row = json.loads(rows.read_text().splitlines()[0])
    return {k: row[k] for k in ("anchors", "basic_units", "by_kind")}


pd.DataFrame({"stock": scan_corpus({}), "edited": scan_corpus(EDITED)}).T

,anchors,basic_units,by_kind
stock,2,0,{'part': 2}
edited,6,4,"{'part': 2, 'section': 4}"


One structuring call (the model again) under the edited configuration, through
`ingest-one`, and the clauses come out as sections with the eIds the `Section` text produced in section 4.

In [7]:
bundle = scratch / "bundle"
run(
    "ingest-one", corpus / "coastal-lights.txt", "--jurisdiction", "xa", "--out", bundle, env=EDITED
)
manifest = json.loads((bundle / "manifest.json").read_text())
print("anchors", manifest["anchors"], "| coverage", manifest["coverage"])
print(" ".join(eids(parse_akn((bundle / "final.akn.xml").read_text()).body)))
shutil.rmtree(scratch)

anchors 6 | coverage {'kind': 'section', 'ratio': 1.0, 'captured': ['1', '2', '3', '4'], 'expected': ['1', '2', '3', '4'], 'missing': [], 'masked': 0, 'unclosed': 0, 'container': {'present': 0, 'found': 0, 'grouping_declared': True}}
part_I part_I__sec_1 part_I__sec_2 part_II part_II__sec_3 part_II__sec_4


The copy above is deleted at the end; nothing in the repo carries the `Clause` rule. To
keep an edit like it, add a jurisdiction of its own with `synthetic` or
`public_reference` set so the wheel carries it: `docs/jurisdictions/adding-a-jurisdiction.md`
is the guide.